# Fourier Transform Visualization Playground

Use this notebook to inspect Fourier-domain preprocessing before committing to training runs.

It downloads the Roboflow shrimp segmentation dataset, rebuilds the deterministic grouped split, samples healthy and diseased images, and visualizes before/after transforms with:

- original image
- transformed image
- absolute difference heatmap
- original FFT magnitude
- transformed FFT magnitude
- optional YOLO polygon mask overlay

Edit only the parameter cells when exploring visual output.

## Run Controls

In [ ]:
from pathlib import Path

SEED = 42
SAMPLES_PER_DISEASE = 2
SAMPLE_SPLIT = 'train'  # train, valid, or test
MAX_IMAGES_TO_SHOW = 8

if Path('/kaggle/working').exists():
    WORK_DIR = Path('/kaggle/working')
elif Path('/content').exists():
    WORK_DIR = Path('/content')
else:
    WORK_DIR = Path.cwd()

DATASET_DIR = WORK_DIR / 'shrimpDisHandSegV2-1'
OUTPUT_DIR = WORK_DIR / 'fourier_visualization_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('WORK_DIR:', WORK_DIR)
print('DATASET_DIR:', DATASET_DIR)
print('SAMPLE_SPLIT:', SAMPLE_SPLIT)
print('SAMPLES_PER_DISEASE:', SAMPLES_PER_DISEASE)

## Install and Import Dependencies

In [ ]:
import importlib.metadata
import subprocess
import sys


def installed_version(package_name):
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return None


for package_name in ['roboflow', 'pandas', 'pyyaml', 'matplotlib', 'opencv-python-headless']:
    if installed_version(package_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

import hashlib
import json
import math
import os
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

print('Imports ready.')

## Download Roboflow Dataset

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY_DIRECT = ''
ROBOFLOW_WORKSPACE = 'lets-try-this'
ROBOFLOW_PROJECT = 'shrimpdishandsegv2'
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = 'yolo26'


def get_roboflow_api_key():
    if ROBOFLOW_API_KEY_DIRECT.strip():
        return ROBOFLOW_API_KEY_DIRECT.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
        if key:
            return key
    except Exception:
        pass
    try:
        from google.colab import userdata
        key = userdata.get('ROBOFLOW_API_KEY')
        if key:
            return key
    except Exception:
        pass
    return os.environ.get('ROBOFLOW_API_KEY', '').strip()


if not (DATASET_DIR / 'data.yaml').exists():
    api_key = get_roboflow_api_key()
    if not api_key:
        raise RuntimeError(
            'Missing Roboflow API key. Add a Kaggle or Colab Secret named ROBOFLOW_API_KEY, '
            'set ROBOFLOW_API_KEY, or temporarily fill ROBOFLOW_API_KEY_DIRECT.'
        )
    rf = Roboflow(api_key=api_key)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    dataset = version.download(ROBOFLOW_FORMAT, location=str(DATASET_DIR))
    print('Downloaded dataset to:', dataset.location)
else:
    print('Dataset already exists:', DATASET_DIR)

base_path = str(DATASET_DIR)
data_yaml_path = str(DATASET_DIR / 'data.yaml')
print('data.yaml:', data_yaml_path)

## Deterministic Grouped Split Helpers

In [ ]:
random.seed(SEED)
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
REBUILD_SPLIT_FROM_ALL_SPLITS = True

SHRIMP_NAME_PATTERN = re.compile(
    r'^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$',
    re.IGNORECASE,
)

for split in ['train', 'valid', 'test']:
    for sub in ['images', 'labels']:
        (DATASET_DIR / split / sub).mkdir(parents=True, exist_ok=True)


def normalize_roboflow_stem(stem):
    stem = re.sub(r'_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    stem = re.sub(r'\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    return stem


def parse_shrimp_group_key(image_name):
    stem = normalize_roboflow_stem(Path(image_name).stem)
    match = SHRIMP_NAME_PATTERN.match(stem)
    if not match:
        return f'unparsed::{Path(image_name).stem}', 'unparsed', None, None
    disease = match.group('disease')
    shrimp_id = match.group('shrimp_id')
    img_num = int(match.group('img_num'))
    group_key = f'{disease.lower()}::{shrimp_id}'
    return group_key, disease, shrimp_id, img_num


def image_files_in_split(split):
    image_dir = DATASET_DIR / split / 'images'
    if not image_dir.exists():
        return []
    return sorted(p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def move_image_and_label(image_path, target_split):
    target_img_dir = DATASET_DIR / target_split / 'images'
    target_lbl_dir = DATASET_DIR / target_split / 'labels'
    target_img_dir.mkdir(parents=True, exist_ok=True)
    target_lbl_dir.mkdir(parents=True, exist_ok=True)

    label_name = f'{image_path.stem}.txt'
    label_src = image_path.parent.parent / 'labels' / label_name
    image_dst = target_img_dir / image_path.name
    label_dst = target_lbl_dir / label_name

    if image_path.resolve() != image_dst.resolve():
        if image_dst.exists():
            raise FileExistsError(f'Duplicate image destination would be overwritten: {image_dst}')
        shutil.move(str(image_path), str(image_dst))

    if label_src.exists():
        if label_src.resolve() != label_dst.resolve():
            if label_dst.exists():
                raise FileExistsError(f'Duplicate label destination would be overwritten: {label_dst}')
            shutil.move(str(label_src), str(label_dst))
    else:
        label_dst.write_text('', encoding='utf-8')


def rebuild_train_pool_from_all_splits():
    all_images = []
    for split in ['train', 'valid', 'test']:
        all_images.extend(image_files_in_split(split))
    for image_path in sorted(all_images):
        move_image_and_label(image_path, 'train')
    return image_files_in_split('train')


def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob('**/*.cache'):
        cache_path.unlink()


def disease_for_group(filenames):
    diseases = []
    for filename in filenames:
        _, disease, _, _ = parse_shrimp_group_key(filename)
        diseases.append(disease)
    counts = Counter(diseases)
    if len(counts) > 1:
        print(f'Warning: group has mixed disease names: {dict(counts)}')
    return counts.most_common(1)[0][0]


def split_one_stratum(items):
    n = len(items)
    train_count = int(TRAIN_RATIO * n)
    val_count = int(VAL_RATIO * n)
    test_count = n - train_count - val_count
    if n >= 3:
        if val_count == 0:
            val_count = 1
            train_count -= 1
        if test_count == 0:
            test_count = 1
            train_count -= 1
    if train_count < 1 and n > 0:
        train_count = 1
    while train_count + val_count + test_count > n:
        train_count -= 1
    test_count = n - train_count - val_count
    return items[:train_count], items[train_count:train_count + val_count], items[train_count + val_count:]


def grouped_stratified_split(group_items):
    strata = defaultdict(list)
    for group_key, filenames in group_items:
        strata[disease_for_group(filenames)].append((group_key, filenames))
    split_to_groups = {'train': [], 'valid': [], 'test': []}
    rng = random.Random(SEED)
    for disease, items in sorted(strata.items()):
        items = sorted(items, key=lambda item: item[0])
        rng.shuffle(items)
        train_items, val_items, test_items = split_one_stratum(items)
        split_to_groups['train'].extend(train_items)
        split_to_groups['valid'].extend(val_items)
        split_to_groups['test'].extend(test_items)
        print(f'  - {disease}: {len(train_items)} train groups, {len(val_items)} valid groups, {len(test_items)} test groups')
    for split in split_to_groups:
        split_to_groups[split] = sorted(split_to_groups[split], key=lambda item: item[0])
    return split_to_groups


def split_grouped_by_shrimp():
    image_paths = rebuild_train_pool_from_all_splits() if REBUILD_SPLIT_FROM_ALL_SPLITS else image_files_in_split('train')
    groups = defaultdict(list)
    disease_counts = Counter()
    for image_path in image_paths:
        group_key, disease, shrimp_id, img_num = parse_shrimp_group_key(image_path.name)
        groups[group_key].append(image_path.name)
        disease_counts[disease] += 1
    split_to_groups = grouped_stratified_split(sorted(groups.items(), key=lambda item: item[0]))
    for split, split_groups in split_to_groups.items():
        for _, filenames in split_groups:
            for filename in filenames:
                move_image_and_label(DATASET_DIR / 'train' / 'images' / filename, split)
    group_to_split = {}
    leakage = []
    for split in ['train', 'valid', 'test']:
        for image_path in image_files_in_split(split):
            group_key, *_ = parse_shrimp_group_key(image_path.name)
            previous_split = group_to_split.setdefault(group_key, split)
            if previous_split != split:
                leakage.append((group_key, previous_split, split, image_path.name))
    if leakage:
        raise RuntimeError(f'Shrimp-level split leakage detected: {leakage[:10]}')
    remove_yolo_label_caches(DATASET_DIR)
    print('Shrimp-level leakage check passed.')
    print('Source filename disease counts before split:', dict(sorted(disease_counts.items())))


split_grouped_by_shrimp()

## Load Class Names and Sample Images

In [ ]:
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)
class_names = data_config.get('names', [])
print('Mask classes:', class_names)


def label_path_for_image(image_path):
    return image_path.parent.parent / 'labels' / f'{image_path.stem}.txt'


def read_label_lines(image_path):
    label_path = label_path_for_image(image_path)
    if not label_path.exists():
        return []
    return [line.strip() for line in label_path.read_text().splitlines() if line.strip()]


def class_ids_for_image(image_path):
    ids = []
    for line in read_label_lines(image_path):
        parts = line.split()
        if parts:
            ids.append(int(float(parts[0])))
    return sorted(set(ids))


def sample_images(split=SAMPLE_SPLIT, per_disease=SAMPLES_PER_DISEASE):
    by_bucket = defaultdict(list)
    for image_path in image_files_in_split(split):
        _, disease, _, _ = parse_shrimp_group_key(image_path.name)
        class_ids = class_ids_for_image(image_path)
        if not class_ids:
            bucket = 'Healthy'
        elif len(class_ids) >= 2:
            bucket = 'WSSV_BG'
        else:
            bucket = class_names[class_ids[0]] if class_ids[0] < len(class_names) else str(class_ids[0])
        by_bucket[bucket].append(image_path)
    rng = random.Random(SEED)
    selected = []
    for bucket in ['Healthy', 'BG', 'WSSV', 'WSSV_BG']:
        items = sorted(by_bucket.get(bucket, []))
        rng.shuffle(items)
        selected.extend(items[:per_disease])
    return selected[:MAX_IMAGES_TO_SHOW]


selected_images = sample_images()
print(f'Selected {len(selected_images)} images from {SAMPLE_SPLIT}:')
for p in selected_images:
    print(' -', p.name, 'labels:', read_label_lines(p)[:2])

## Fourier Transform and Visualization Functions

Keep function definitions here. Edit parameters in the later cells.

In [ ]:
def ensure_rgb(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def ensure_bgr(img_rgb):
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)


def lowpass_filter(shape_hw, sigma):
    height, width = shape_hw
    y = np.arange(height, dtype=np.float32) - height / 2.0
    x = np.arange(width, dtype=np.float32) - width / 2.0
    xx, yy = np.meshgrid(x, y)
    return np.exp(-(xx * xx + yy * yy) / (2.0 * float(sigma) * float(sigma))).astype(np.float32)


def fft_filter_channel(channel, frequency_filter):
    freq = np.fft.fftshift(np.fft.fft2(channel.astype(np.float32)))
    filtered = freq * frequency_filter
    return np.fft.ifft2(np.fft.ifftshift(filtered)).real


def highpass_boost(img_rgb, sigma=50, alpha=0.1):
    img = img_rgb.astype(np.float32)
    lp = lowpass_filter(img.shape[:2], sigma)
    out = []
    for c in range(img.shape[2]):
        low = fft_filter_channel(img[:, :, c], lp)
        high = img[:, :, c] - low
        out.append(img[:, :, c] + float(alpha) * high)
    return np.clip(np.stack(out, axis=2), 0, 255).astype(np.uint8)


def highfreq_damping(img_rgb, sigma=50, alpha=0.1):
    img = img_rgb.astype(np.float32)
    lp = lowpass_filter(img.shape[:2], sigma)
    out = []
    for c in range(img.shape[2]):
        low = fft_filter_channel(img[:, :, c], lp)
        high = img[:, :, c] - low
        out.append(img[:, :, c] - float(alpha) * high)
    return np.clip(np.stack(out, axis=2), 0, 255).astype(np.uint8)


def bandpass_boost(img_rgb, low_sigma=12, high_sigma=60, alpha=0.2):
    img = img_rgb.astype(np.float32)
    lp_low = lowpass_filter(img.shape[:2], low_sigma)
    lp_high = lowpass_filter(img.shape[:2], high_sigma)
    out = []
    for c in range(img.shape[2]):
        low_narrow = fft_filter_channel(img[:, :, c], lp_low)
        low_wide = fft_filter_channel(img[:, :, c], lp_high)
        band = low_wide - low_narrow
        out.append(img[:, :, c] + float(alpha) * band)
    return np.clip(np.stack(out, axis=2), 0, 255).astype(np.uint8)


def lowfreq_flatten(img_rgb, sigma=100, beta=0.3):
    img = img_rgb.astype(np.float32)
    lp = lowpass_filter(img.shape[:2], sigma)
    out = []
    for c in range(img.shape[2]):
        channel = img[:, :, c]
        low = fft_filter_channel(channel, lp)
        mean = channel.mean()
        out.append(channel - float(beta) * (low - mean))
    return np.clip(np.stack(out, axis=2), 0, 255).astype(np.uint8)


def homomorphic_filter(img_rgb, sigma=50, gamma_low=0.7, gamma_high=1.2):
    img = img_rgb.astype(np.float32)
    lp = lowpass_filter(img.shape[:2], sigma)
    gain = float(gamma_low) + (float(gamma_high) - float(gamma_low)) * (1.0 - lp)
    out = []
    for c in range(img.shape[2]):
        log_ch = np.log1p(img[:, :, c])
        filtered = fft_filter_channel(log_ch, gain)
        out.append(np.expm1(filtered))
    return np.clip(np.stack(out, axis=2), 0, 255).astype(np.uint8)


def fft_magnitude_image(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    mag = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(gray))))
    lo, hi = np.percentile(mag, [1, 99])
    mag = np.clip((mag - lo) / max(1e-6, hi - lo), 0, 1)
    return mag


def absolute_difference_heatmap(original_rgb, transformed_rgb):
    diff = np.abs(original_rgb.astype(np.float32) - transformed_rgb.astype(np.float32)).mean(axis=2)
    vmax = np.percentile(diff, 99)
    return np.clip(diff / max(1e-6, vmax), 0, 1)


def yolo_polygon_to_pixels(values, width, height):
    coords = np.array(values, dtype=np.float32).reshape(-1, 2)
    coords[:, 0] *= width
    coords[:, 1] *= height
    return coords.astype(np.int32)


def overlay_yolo_masks(img_rgb, label_lines, alpha=0.35):
    overlay = img_rgb.copy()
    height, width = img_rgb.shape[:2]
    colors = [(255, 60, 60), (60, 180, 255), (80, 220, 120), (240, 200, 60)]
    for line in label_lines:
        parts = line.split()
        if len(parts) < 7:
            continue
        class_id = int(float(parts[0]))
        coords = [float(v) for v in parts[1:]]
        if len(coords) % 2 != 0:
            continue
        pts = yolo_polygon_to_pixels(coords, width, height)
        color = colors[class_id % len(colors)]
        mask_layer = overlay.copy()
        cv2.fillPoly(mask_layer, [pts], color)
        overlay = cv2.addWeighted(mask_layer, alpha, overlay, 1 - alpha, 0)
        cv2.polylines(overlay, [pts], isClosed=True, color=color, thickness=2)
    return overlay


def visualize_transform(image_paths, transform_fn, title, max_images=MAX_IMAGES_TO_SHOW, show_masks=True, save_dir=None):
    rows = min(len(image_paths), max_images)
    if rows == 0:
        print('No images selected.')
        return
    fig, axes = plt.subplots(rows, 6, figsize=(22, 3.8 * rows))
    if rows == 1:
        axes = np.expand_dims(axes, axis=0)
    for row_idx, image_path in enumerate(image_paths[:rows]):
        img_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if img_bgr is None:
            continue
        img_rgb = ensure_rgb(img_bgr)
        transformed = transform_fn(img_rgb)
        labels = read_label_lines(image_path)
        original_show = overlay_yolo_masks(img_rgb, labels) if show_masks else img_rgb
        transformed_show = overlay_yolo_masks(transformed, labels) if show_masks else transformed
        diff = absolute_difference_heatmap(img_rgb, transformed)
        fft_orig = fft_magnitude_image(img_rgb)
        fft_trans = fft_magnitude_image(transformed)
        _, disease, shrimp_id, img_num = parse_shrimp_group_key(image_path.name)
        row_title = f'{disease} | {image_path.name}'
        panels = [
            (original_show, 'Original + mask' if show_masks else 'Original', None),
            (transformed_show, 'Transformed + mask' if show_masks else 'Transformed', None),
            (diff, 'Abs diff', 'magma'),
            (fft_orig, 'FFT original', 'viridis'),
            (fft_trans, 'FFT transformed', 'viridis'),
            (np.abs(fft_trans - fft_orig), 'FFT abs diff', 'magma'),
        ]
        for col_idx, (panel, panel_title, cmap) in enumerate(panels):
            ax = axes[row_idx, col_idx]
            if cmap:
                ax.imshow(panel, cmap=cmap)
            else:
                ax.imshow(panel)
            ax.set_title(panel_title, fontsize=10)
            ax.axis('off')
        axes[row_idx, 0].set_ylabel(row_title, fontsize=9)
    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        safe_title = re.sub(r'[^a-zA-Z0-9_\-]+', '_', title).strip('_').lower()
        out_path = save_dir / f'{safe_title}.png'
        fig.savefig(out_path, dpi=160, bbox_inches='tight')
        print('Saved figure:', out_path)
    plt.show()

## Visualize High-Pass Boost

Edit `HP_SIGMA` and `HP_ALPHA`, then rerun this cell.

In [ ]:
HP_SIGMA = 50
HP_ALPHA = 0.10

visualize_transform(
    selected_images,
    lambda img: highpass_boost(img, sigma=HP_SIGMA, alpha=HP_ALPHA),
    title=f'High-pass boost sigma={HP_SIGMA}, alpha={HP_ALPHA}',
    save_dir=OUTPUT_DIR,
)

## Visualize High-Frequency Damping

Edit `DAMP_SIGMA` and `DAMP_ALPHA`, then rerun this cell.

In [ ]:
DAMP_SIGMA = 50
DAMP_ALPHA = 0.10

visualize_transform(
    selected_images,
    lambda img: highfreq_damping(img, sigma=DAMP_SIGMA, alpha=DAMP_ALPHA),
    title=f'High-frequency damping sigma={DAMP_SIGMA}, alpha={DAMP_ALPHA}',
    save_dir=OUTPUT_DIR,
)

## Visualize Band-Pass Boost

Edit `BP_LOW_SIGMA`, `BP_HIGH_SIGMA`, and `BP_ALPHA`, then rerun this cell.

In [ ]:
BP_LOW_SIGMA = 12
BP_HIGH_SIGMA = 60
BP_ALPHA = 0.20

visualize_transform(
    selected_images,
    lambda img: bandpass_boost(img, low_sigma=BP_LOW_SIGMA, high_sigma=BP_HIGH_SIGMA, alpha=BP_ALPHA),
    title=f'Band-pass boost low={BP_LOW_SIGMA}, high={BP_HIGH_SIGMA}, alpha={BP_ALPHA}',
    save_dir=OUTPUT_DIR,
)

## Visualize Low-Frequency Flattening

Edit `FLAT_SIGMA` and `FLAT_BETA`, then rerun this cell.

In [ ]:
FLAT_SIGMA = 100
FLAT_BETA = 0.30

visualize_transform(
    selected_images,
    lambda img: lowfreq_flatten(img, sigma=FLAT_SIGMA, beta=FLAT_BETA),
    title=f'Low-frequency flatten sigma={FLAT_SIGMA}, beta={FLAT_BETA}',
    save_dir=OUTPUT_DIR,
)

## Visualize Homomorphic Filtering

Edit `HOMO_SIGMA`, `HOMO_GAMMA_LOW`, and `HOMO_GAMMA_HIGH`, then rerun this cell.

In [ ]:
HOMO_SIGMA = 50
HOMO_GAMMA_LOW = 0.70
HOMO_GAMMA_HIGH = 1.20

visualize_transform(
    selected_images,
    lambda img: homomorphic_filter(
        img,
        sigma=HOMO_SIGMA,
        gamma_low=HOMO_GAMMA_LOW,
        gamma_high=HOMO_GAMMA_HIGH,
    ),
    title=f'Homomorphic sigma={HOMO_SIGMA}, gamma_low={HOMO_GAMMA_LOW}, gamma_high={HOMO_GAMMA_HIGH}',
    save_dir=OUTPUT_DIR,
)

## Compare Multiple Transforms on One Image

Use this for quick side-by-side inspection of one sample.

In [ ]:
COMPARE_IMAGE_INDEX = 0
image_path = selected_images[COMPARE_IMAGE_INDEX]
img_rgb = ensure_rgb(cv2.imread(str(image_path), cv2.IMREAD_COLOR))
labels = read_label_lines(image_path)

variants = [
    ('Original', img_rgb),
    ('High-pass a0.1 s50', highpass_boost(img_rgb, sigma=50, alpha=0.10)),
    ('High-pass a0.3 s50', highpass_boost(img_rgb, sigma=50, alpha=0.30)),
    ('Damping a0.1 s50', highfreq_damping(img_rgb, sigma=50, alpha=0.10)),
    ('Band-pass 12-60 a0.2', bandpass_boost(img_rgb, low_sigma=12, high_sigma=60, alpha=0.20)),
    ('Low-flat s100 b0.3', lowfreq_flatten(img_rgb, sigma=100, beta=0.30)),
    ('Homomorphic', homomorphic_filter(img_rgb, sigma=50, gamma_low=0.70, gamma_high=1.20)),
]

cols = 4
rows = math.ceil(len(variants) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(18, 4.5 * rows))
axes = np.array(axes).reshape(rows, cols)
for ax in axes.ravel():
    ax.axis('off')
for idx, (name, variant) in enumerate(variants):
    ax = axes.ravel()[idx]
    ax.imshow(overlay_yolo_masks(variant, labels))
    ax.set_title(name)
    ax.axis('off')
fig.suptitle(f'One-image comparison: {image_path.name}', fontsize=16)
plt.tight_layout()
out_path = OUTPUT_DIR / f'compare_one_image_{image_path.stem}.png'
fig.savefig(out_path, dpi=160, bbox_inches='tight')
print('Saved figure:', out_path)
plt.show()